In [8]:
"""
B.TECH mobile phone scraper — v3, hits the real search API directly.

Confirmed via browser DevTools (2026-07):
    POST https://retail-online-prod.btech.com/api/v1/green/discovery/api/v1/products/search
    Content-Type: application/json
    Body: {"page": N, "page_size": 20, "filters": {"categories": "mobile-phones", "in_stock": true}}

This replaces the earlier HTML-embedded-payload approach (btech_parser.py) —
no longer needed for B.TECH, but kept in the repo in case another site on
your list uses a similar Next.js RSC-embedding pattern.

Install:
    pip install requests --break-system-packages
"""

import csv
import datetime as dt
import time
from dataclasses import dataclass, asdict, field
from typing import Optional

import requests


In [22]:
API_URL = "https://retail-online-prod.btech.com/api/v1/green/discovery/api/v1/products/search"
OUTPUT_CSV = "btech_raw.csv"
PAGE_SIZE = 20
MAX_PAGES = 39          # confirmed total_pages from the API's own response
REQUEST_DELAY_SEC = 1.0  # be polite

from dotenv import load_dotenv
import os

load_dotenv("secrets.env")

BTECH_AUTH_TOKEN = os.getenv("BTECH_AUTH_TOKEN")

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Accept-Language": "en",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36"
    ),
    "Origin": "https://btech.com",
    "Referer": "https://btech.com/",
    "X-Platform": "web",
    # Guest/anonymous access token — NOT a personal login credential.
    # is_guest: true in the JWT payload. Issued with a ~1 year expiry.
    # If this starts returning 401 again later, it's expired — repeat the
    # DevTools capture (Network tab, same request) to grab a fresh one.
    "Authorization": f"Bearer {BTECH_AUTH_TOKEN}",

}


In [10]:
@dataclass
class Product:
    source_site: str = "btech"
    scraped_at: str = field(default_factory=lambda: dt.datetime.now().isoformat())
    product_url: str = ""
    image_url: str = ""
    brand: str = ""
    model_name: str = ""
    variant_id: str = ""
    product_id: str = ""
    price_egp: Optional[float] = None
    original_price_egp: Optional[float] = None
    in_stock: Optional[bool] = None
    is_best_seller: Optional[bool] = None
    is_new_arrival: Optional[bool] = None
    seller_name: str = ""
    category_l3: str = ""


def _to_product(item: dict) -> Product:
    price = item.get("price", {}) or {}
    return Product(
        product_url=f"https://btech.com/en/p/{item.get('slug', '')}",
        image_url=item.get("thumbnail_url", ""),
        brand=item.get("brand", ""),
        model_name=item.get("name", ""),
        variant_id=item.get("variant_id", ""),
        product_id=item.get("product_id", ""),
        price_egp=price.get("final_price"),
        original_price_egp=price.get("base_price"),
        in_stock=item.get("is_in_stock"),
        is_best_seller=item.get("is_best_seller"),
        is_new_arrival=item.get("is_new_arrival"),
        seller_name=item.get("seller_name", ""),
        category_l3=(item.get("categories") or {}).get("l3", {}).get("name", ""),
    )


In [11]:
def _fetch_page(page_num: int) -> dict:
    payload = {
        "page": page_num,
        "page_size": PAGE_SIZE,
        "filters": {"categories": "mobile-phones", "in_stock": True},
    }
    resp = requests.post(API_URL, json=payload, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.json()


In [12]:
def main():
    all_products: list[Product] = []
    seen_variant_ids: set[str] = set()

    for page_num in range(1, MAX_PAGES + 1):
        print(f"Fetching page {page_num}/{MAX_PAGES}...")
        data = _fetch_page(page_num)

        items = data.get("items", [])
        if not items:
            print("  No items returned — stopping.")
            break

        new_count = 0
        for item in items:
            vid = item.get("variant_id")
            if vid in seen_variant_ids:
                continue
            seen_variant_ids.add(vid)
            all_products.append(_to_product(item))
            new_count += 1

        total_pages = data.get("total_pages")
        total_items = data.get("total_items")
        print(
            f"  -> {new_count} new products (running total: {len(all_products)}) "
            f"| total_items={total_items} total_pages={total_pages}"
        )

        if total_pages and page_num >= total_pages:
            print("  Reached last page.")
            break

        time.sleep(REQUEST_DELAY_SEC)

    if all_products:
        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(asdict(all_products[0]).keys()))
            writer.writeheader()
            for p in all_products:
                writer.writerow(asdict(p))
        print(f"\nSaved {len(all_products)} products to {OUTPUT_CSV}")
    else:
        print("\nNo products scraped.")


if __name__ == "__main__":
    main()

Fetching page 1/39...
  -> 20 new products (running total: 20) | total_items=699 total_pages=35
Fetching page 2/39...
  -> 20 new products (running total: 40) | total_items=699 total_pages=35
Fetching page 3/39...
  -> 20 new products (running total: 60) | total_items=699 total_pages=35
Fetching page 4/39...
  -> 20 new products (running total: 80) | total_items=699 total_pages=35
Fetching page 5/39...


KeyboardInterrupt: 

In [15]:
import pandas as pd

df = pd.read_csv("phone_cleaned.csv")

pro_url = df['product_url']

print(pro_url)

0      https://btech.com/en/p/82f1a686-4373-43d5-b79c...
1      https://btech.com/en/p/760705ec-59eb-4196-a844...
2      https://btech.com/en/p/95ea8720-6e24-436b-bc06...
3      https://btech.com/en/p/ba7338f2-0084-4d9b-b80b...
4      https://btech.com/en/p/012251d6-c872-47db-be03...
                             ...                        
658    https://btech.com/en/p/8b563fdf-165d-4425-a37a...
659    https://btech.com/en/p/6a9f7b89-edd1-40e5-826b...
660    https://btech.com/en/p/samsung-galaxy-a17-128g...
661    https://btech.com/en/p/d84b86e7-bcd1-4da1-808e...
662    https://btech.com/en/p/42a29915-7634-481e-9a21...
Name: product_url, Length: 663, dtype: str


In [17]:
from bs4 import BeautifulSoup
import requests 
import re
import time


In [13]:
def fetch_product_specs(product_url: str, headers: dict) -> dict:
    resp = requests.get(product_url, headers=headers, timeout=20)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    result = {"key_features_text": "", "specifications": {}}

    # --- Key features ---
    kf_heading = soup.find(lambda tag: tag.name in ("h2", "h3") and "Key features" in tag.get_text())
    if kf_heading:
        texts = []
        for sib in kf_heading.find_next_siblings():
            if sib.name in ("h2", "h3"):
                break
            texts.append(sib.get_text(separator="\n", strip=True))
        result["key_features_text"] = "\n".join(t for t in texts if t)

    # --- Specifications table ---
    spec_table = soup.find("table", class_="w-full")
    if spec_table:
        tbody = spec_table.find("tbody")
        if tbody:
            rows = tbody.find_all("tr")
            for row in rows:
                tds = row.find_all("td")
                if len(tds) == 2:
                    key = tds[0].get_text(strip=True)
                    val = tds[1].get_text(strip=True)
                    if key:
                        result["specifications"][key] = val

    return result

In [18]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en",
}
url = pro_url[0]  # Example: take the first product URL
fetched_specs = fetch_product_specs(url, HEADERS)
print(fetched_specs)

{'key_features_text': "Brand: Infinix\nModel Name: Smart 20\n\nDisplay:\n- 6.78 inch, IPS LCD HD+ 720x1576\n- 45Hz/60Hz/90Hz/120Hz refresh rate\n-560 nits (TYP), 700 nits (HBM Brightness)\n\nProcessor: MediaTek Helio G81 Ultimate- Octa Core\n\nRear Camera:\n-8MP, f/2.0, 1/4'' sensor, 1.12um, AF \n-Rear Dual Flash\n-AI Cam, Video, Dual Video, Portrait, Time-Lapse, Super Night, Pro, Panorama, Documents\n-Video 2K 30FPS/1080P,30FPS/720P 30FPS\n\nFront Camera: 8MP, f/2.0, 1/4'' sensor, 1.12um, FF\n\nBattery Capacity: 5200mAh, 10W Charger, 15W Supported\nNumber of sim cards: 2\n\nConnectivity:\n-Wi-Fi 802.11\n-Bluetooth\n-USB Type-C, OTG\n- 3.5 Jack\n-FM Radio\n\nOperating System:  XOS16, Powered by Android 16\n\nOther Features:\n-G-sensor, e-compass, gyroscope, light sensor, proximity sensor\n-Fingerprint sensor\n-IP64 splash, water and dust resistance", 'specifications': {'Network': '4G', 'Sim Type': 'Nano-SIM', 'Port Type': 'Type C', 'Display Type': 'IPS LCD', 'Charging Type': 'Wired', '

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en",
}


def fetch_product_specs(product_url: str, headers: dict) -> dict:
    resp = requests.get(product_url, headers=headers, timeout=20)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    result = {
        "key_features_text": "",
        "specifications": {}
    }

    # --- Key features ---
    kf_heading = soup.find(
        lambda tag: (
            tag.name in ("h2", "h3")
            and "Key features" in tag.get_text()
        )
    )

    if kf_heading:
        texts = []

        for sib in kf_heading.find_next_siblings():
            if sib.name in ("h2", "h3"):
                break

            text = sib.get_text(separator="\n", strip=True)

            if text:
                texts.append(text)

        result["key_features_text"] = "\n".join(texts)

    # --- Specifications table ---
    spec_table = soup.find("table", class_="w-full")

    if spec_table:
        tbody = spec_table.find("tbody")

        if tbody:
            rows = tbody.find_all("tr")

            for row in rows:
                tds = row.find_all("td")

                if len(tds) == 2:
                    key = tds[0].get_text(strip=True)
                    val = tds[1].get_text(strip=True)

                    if key:
                        result["specifications"][key] = val

    return result


In [ ]:

scraped_products = []

for index, row in df.iterrows():

    print(f"[{index + 1}/{len(df)}] Scraping: {pro_url[index]}")

    try:
        specs = fetch_product_specs(pro_url[index], HEADERS)

        # Start with information from the original dataset
        product_data = row.to_dict()

        # Add key features
        product_data["key_features_text"] = specs["key_features_text"]

        # Add every specification as a column
        for key, value in specs["specifications"].items():
            product_data[key] = value

        scraped_products.append(product_data)

    except Exception as e:
        print(f"ERROR: {e}")

        # Keep the original row even if scraping fails
        product_data = row.to_dict()
        product_data["key_features_text"] = ""

        scraped_products.append(product_data)

    # Be polite to the website
    time.sleep(1)


# --------------------------------------------------
# Create final dataset
# --------------------------------------------------

final_df = pd.DataFrame(scraped_products)

# Save CSV
final_df.to_csv(
    "phone_with_specs.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nDone!")
print(f"Saved {len(final_df)} products.")
print("Output: phone_with_specs.csv")

[1/663] Scraping: https://btech.com/en/p/82f1a686-4373-43d5-b79c-19c02f8408ff
[2/663] Scraping: https://btech.com/en/p/760705ec-59eb-4196-a844-687dfb44e8c6
[3/663] Scraping: https://btech.com/en/p/95ea8720-6e24-436b-bc06-a4cc2f09f5aa
[4/663] Scraping: https://btech.com/en/p/ba7338f2-0084-4d9b-b80b-3664df6ac7dc
[5/663] Scraping: https://btech.com/en/p/012251d6-c872-47db-be03-c4e12687f205
[6/663] Scraping: https://btech.com/en/p/xiaomi-redmi-15c-256gb-8gb-ram-4g-dual-sim-midnight-gray
[7/663] Scraping: https://btech.com/en/p/samsung-galaxy-a17-256gb-8gb-ram-4g-dual-sim-black
[8/663] Scraping: https://btech.com/en/p/5b726b61-7507-404f-9447-4dc0305f67f2
[9/663] Scraping: https://btech.com/en/p/xiaomi-redmi-15c-256gb-8gb-ram-4g-dual-sim-moonlight-blue
[10/663] Scraping: https://btech.com/en/p/864d20c8-9b01-4aac-ab53-f6ce0ef7321b
[11/663] Scraping: https://btech.com/en/p/770d7133-3838-4581-8d7f-ee3563651467
[12/663] Scraping: https://btech.com/en/p/a4e2507d-7968-47c2-acd6-ed429c18a3ff
[13/66

In [ ]:
import pandas as pd
final_df = pd.read_csv("phone_with_specs.csv")
final_df["key_features_text"].head(1)

0    Brand: Infinix\nModel Name: Smart 20\n\nDispla...
Name: key_features_text, dtype: str

In [19]:
import pandas as pd
import json


def create_features_json(
    csv_file="phone_with_specs.csv",
    json_file="phone_features.json"
):
    # Load CSV
    df = pd.read_csv(csv_file)

    # Get key features column
    separate_features = df["key_features_text"]

    all_products = []

    # Loop through every product
    for feature_text in separate_features:

        if pd.isna(feature_text):
            all_products.append({})
            continue

        lines = feature_text.splitlines()

        product_features = {}

        current_feature = None
        current_values = []

        for line in lines:

            line = line.strip()

            # Skip empty lines
            if not line:
                continue

            # If line starts with "-"
            if line.startswith("-"):

                value = line.lstrip("-").strip()

                if current_feature and value:
                    current_values.append(value)

            # If line contains ":"
            elif ":" in line:

                # Save previous feature
                if current_feature:
                    product_features[current_feature] = "; ".join(
                        current_values
                    )

                # Create new feature
                current_feature, value = line.split(":", 1)

                current_feature = current_feature.strip()
                value = value.strip()

                current_values = []

                if value:
                    current_values.append(value)

            # Continuation of previous feature
            else:

                if current_feature:
                    current_values.append(line)

        # Save last feature
        if current_feature:
            product_features[current_feature] = "; ".join(
                current_values
            )

        all_products.append(product_features)

    # Save everything to JSON
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(
            all_products,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"JSON file created: {json_file}")
    print(f"Products processed: {len(all_products)}")

In [20]:
create_features_json()


JSON file created: phone_features.json
Products processed: 663


In [21]:
import pandas as pd
import json

# Load both datasets
df = pd.read_csv("phone_cleaned.csv")

with open("phone_features.json", "r", encoding="utf-8") as f:
    features = json.load(f)

# Check that both have the same number of products
print("CSV rows:", len(df))
print("JSON products:", len(features))

if len(df) != len(features):
    raise ValueError("CSV and JSON have different numbers of rows!")

# Add the extracted features to the CSV dataframe
features_df = pd.DataFrame(features)

# Reset indexes to make sure they match
df = df.reset_index(drop=True)
features_df = features_df.reset_index(drop=True)

# Combine them
linked_df = pd.concat(
    [df, features_df],
    axis=1
)

# Save
linked_df.to_csv(
    "phones_linked.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Created phones_linked.csv")

CSV rows: 663
JSON products: 663
Created phones_linked.csv
